# Illustrating CLIP: One Shared Space for Images and Text

**CLIP** (Contrastive Language-Image Pre-training) is not a chat model. It's a pair of encoders — one for images, one for text — trained so that matching image/caption pairs land close together in a shared embedding space, and mismatched pairs land far apart.

We this idea, we can do:

- **Zero-shot classification** — classify an image against arbitrary text labels you invent at runtime, with no fine-tuning.
- **Semantic image search** — find images by describing them in words.
- **Similarity & retrieval** — rank captions for an image, or images for an image.

The method: encode image to a vector, encode text to a vector, compare with cosine similarity.

We illustrate this below with HuggingFace `transformers`. The [`open_clip`](https://github.com/mlfoundations/open_clip) library is a popular alternative with more model variants — a short example is at the end.

## Load the libraries, model, and processor

`transformers` and `torch` are the model; the rest are for loading and plotting images. The first model load downloads weights from the HuggingFace Hub.

The "processor" bundles two things: an image preprocessor (resize, crop, normalize) and a tokenizer (turn text into token IDs). The "model" holds both encoders.

`openai/clip-vit-base-patch32` is small and fast — good for our demo. Larger, stronger checkpoints include `openai/clip-vit-large-patch14` and the LAION-trained `laion/CLIP-ViT-H-14-laion2B-s32B-b79K` (LAION: Large-scale Artificial Intelligence Open Network, a German non-profit organization dedicated to democratizing artificial intelligence by releasing open-source datasets, models, and tools to the public).

In [ ]:
import torch
from transformers import CLIPModel, CLIPProcessor

device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
MODEL_NAME = "openai/clip-vit-base-patch32"

processor = CLIPProcessor.from_pretrained(MODEL_NAME)

model = CLIPModel.from_pretrained(MODEL_NAME).to(device)
model.eval()  # we only use the model here for inference

print(f"Loaded {MODEL_NAME} on {device}")
print(f"Embedding dimension: {model.config.projection_dim}")

## Set up sample images

We use a few labeled sample images that ship inside `scikit-image` — a cat, a cup of coffee, an astronaut,
and a rocket launch.

To use your own images instead, replace the dict values with `Image.open("your_file.jpg").convert("RGB")`.

In [ ]:
from PIL import Image
from skimage import data

images = {
    "cat":       Image.fromarray(data.chelsea()),    # a tabby cat
    "coffee":    Image.fromarray(data.coffee()),     # a cup of coffee
    "astronaut": Image.fromarray(data.astronaut()),  # a person (astronaut)
    "rocket":    Image.fromarray(data.rocket()),     # a rocket launching
    "grogu":     Image.open('grogu_clarinet.jpg').convert('RGB'),     # grogu holding a clarinet
}

# Preview them
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, len(images), figsize=(3 * len(images), 3))
for ax, (name, img) in zip(axes, images.items()):
    ax.imshow(img); ax.set_title(name); ax.axis("off")
plt.tight_layout(); plt.show()

## Zero-shot classification

Pick an image, invent a list of candidate text labels, and ask CLIP which label best matches. There's no training step — the labels can be anything.

Under the hood: the processor encodes the image and each label, the model computes image-vs-text similarities (`logits_per_image`), and a softmax turns those into probabilities.

In [ ]:
labels = ["a cat", "a cup of coffee", "an astronaut", "a rocket", 
          "a mountain landscape", 
          "a clarinet", "a jedi", "grogu", "grogu and a clarinet", "darth vader and a clarinet"]
target = images["cat"]
# target = images["grogu"]

inputs = processor(text=labels, images=target, return_tensors="pt", padding=True).to(device)

with torch.no_grad():
    outputs = model(**inputs)

# logits_per_image: similarity of the image to each label (already temperature-scaled)
probs = outputs.logits_per_image.softmax(dim=1).squeeze()
# probs = outputs.logits_per_image.softmax(dim=1).squeeze().cpu().numpy()

for label, p in sorted(zip(labels, probs), key=lambda x: -x[1]):
    print(f"{p:6.1%}  {label}")

Let's visualize that as a bar chart next to the image.

In [ ]:
fig, (ax_img, ax_bar) = plt.subplots(1, 2, figsize=(9, 3.5))
ax_img.imshow(target); ax_img.axis("off"); ax_img.set_title("input")

order = probs.argsort()
ax_bar.barh([labels[i] for i in order], [probs[i] for i in order])
ax_bar.set_xlim(0, 1); ax_bar.set_xlabel("probability")
ax_bar.set_title("zero-shot scores")
plt.tight_layout(); plt.show()

## Looking at the embeddings

`logits_per_image` is convenient, but the real objects are the embedding vectors. To look at them directly, the standard recipe is:

1. Encode to raw feature vectors.
2. L2-normalize them (so each sits on the unit sphere).
3. Compare with a dot product — which, for unit vectors, is cosine similarity.

`get_image_features` and `get_text_features` give us the vectors.

[Technical detail:  In transformers 4.x, `get_image_features()` returned a plain `torch.FloatTensor` directly, but in transformers 5.x, it returns a `BaseModelOutputWithPooling` object (a dataclass) rather than a raw tensor.  Hence we use the additional `.pooler_output`]

In [ ]:
import torch.nn.functional as F

def embed_images(pil_images):
    inputs = processor(images=list(pil_images), return_tensors="pt").to(device)
    with torch.no_grad():
        feats = model.get_image_features(**inputs).pooler_output
    return F.normalize(feats, dim=-1)

def embed_texts(texts):
    inputs = processor(text=list(texts), return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        feats = model.get_text_features(**inputs).pooler_output
    return F.normalize(feats, dim=-1)

img_vecs = embed_images(images.values())
print("image embeddings shape:", tuple(img_vecs.shape))   # (n_images, dim)

vec = embed_texts(["a photograph of a cat"])
print("text  embedding  shape:", tuple(vec.shape))         # (1, dim)
print("L2 norm (should be ~1):", vec.norm().item())

## Getting the [image, text] similarity matrix

The similarity matrix illuminates CLIP best. After encoding every image and every caption, the full matrix of cosine similarities shows us the correspondence between images and text. The matching pairs should light up along the diagonal.

In [ ]:
import numpy as np

captions = [
    "a photo of a cat",
    "a cup of coffee",
    "an astronaut",
    "a rocket launching",
    "grogu and a clarinet", 
    # "a mountain landscape",
    # "a clarinet", 
    # "a jedi", 
    # "grogu", 
    # "darth vader and a clarinet"
]
img_names = list(images.keys())

I = embed_images(images.values())          # (n_img, dim)
T = embed_texts(captions)                  # (n_txt, dim)
sim = (I @ T.T).cpu().numpy()              # cosine similarity matrix

fig, ax = plt.subplots(figsize=(7.5, 4.5))
im = ax.imshow(sim, vmin=0, vmax=sim.max(), cmap="viridis")
ax.set_xticks(range(len(captions)), captions, rotation=30, ha="right")
ax.set_yticks(range(len(img_names)), img_names)
for i in range(sim.shape[0]):
    for j in range(sim.shape[1]):
        ax.text(j, i, f"{sim[i, j]:.2f}", ha="center", va="center",
                color="white" if sim[i, j] < sim.max() * 0.6 else "black")
ax.set_title("cosine similarity: images vs captions")
fig.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout(); plt.show()

The brightest cell in each row should be the caption that actually describes that image. That diagonal dominance is the contrastive training objective.

## Semantic image search (text to image)

Because images and text share one space, "search" can be done by embedding the query text, then ranking images by similarity. (This may feel similar to what we did before with RAG).

Note the query doesn't have to be a class name — it can be a free-form description.

In [ ]:
def search(query, image_vecs, names, top_k=3):
    q = embed_texts([query])               # (1, dim)
    scores = (image_vecs @ q.T).squeeze(1).cpu().numpy()
    ranked = scores.argsort()[::-1][:top_k]
    return [(names[i], float(scores[i])) for i in ranked]

img_vecs = embed_images(images.values())
names = list(images.keys())

for query in ["something you can drink", "an animal", "outer space", "musical instrument"]:
    results = search(query, img_vecs, names)
    print(f"Query: {query!r}")
    for name, score in results:
        print(f"    {score:.3f}  {name}")
    print()

## Two things that shift the numbers

There are two details that matter here:

**1. The temperature is not optional.** To turn similarities into probabilities, CLIP multiplies the cosine scores by a large learned temperature (`logit_scale`, which exponentiates to \~100) before the softmax. CLIP's cosine similarities all sit in a narrow band (\~0.2–0.3), so if you softmax them raw, every class comes out near chance (\~uniform) — the prediction looks like noise. At the very top, we avoided this by using `logits_per_image`, which bakes the temperature in. When you compute probabilities from your own embeddings, you must reapply it.

**2. Prompt wording helps — modestly, and mostly in aggregate.** A bare label like `"cat"` is out-of-distribution versus the web captions CLIP trained on, so a template like `"a photo of a {}"` tends to help, and averaging several
templates (prompt ensembling) helps a bit more. But the gain is a few points of accuracy averaged over many classes and hard cases — not a dramatic swing on easy, visually distinct ones. On this toy set, expect all the scaled columns below to be confidently correct and close to each other; the dramatic difference is raw-vs-scaled, not template-vs-ensemble.

In [ ]:
target = images["coffee"]
# target = images["grogu"]
target_vec = embed_images([target])       # (1, dim)


TEMPLATES = [
    "a photo of a {}.",
    "a blurry photo of a {}.",
    "a close-up photo of a {}.",
    "a bright photo of a {}.",
    "a cropped photo of a {}.",
]

def ensemble_text_embedding(classname):
    prompts = [t.format(classname) for t in TEMPLATES]
    vecs = embed_texts(prompts)            # (n_templates, dim)
    mean = vecs.mean(dim=0, keepdim=True)  # average, then renormalize
    return F.normalize(mean, dim=-1)

classes = ["cat", "coffee", "astronaut", "rocket", "grogu"]

bare = embed_texts(classes)
templated = embed_texts([f"a photo of a {c}" for c in classes])
ensemble = torch.cat([ensemble_text_embedding(c) for c in classes], dim=0)

# CLIP's learned temperature — the piece a raw cosine softmax leaves out.
# .item() detaches it to a plain float so later .numpy() calls are safe.
logit_scale = model.logit_scale.exp().item()

def class_probs(text_vecs, scaled=True):
    sims = target_vec @ text_vecs.T        # (1, n_classes) cosine similarities
    if scaled:
        sims = logit_scale * sims          # <-- apply temperature
    return sims.softmax(dim=1).squeeze().cpu().numpy()

raw_unscaled = class_probs(bare, scaled=False)   # the near-uniform "noise"
bare_scaled  = class_probs(bare)
tmpl_scaled  = class_probs(templated)
ens_scaled   = class_probs(ensemble)

print(f"{'class':<10}{'raw cosine':>12}{'bare+temp':>12}{'template':>12}{'ensemble':>12}")
for i, c in enumerate(classes):
    print(f"{c:<10}{raw_unscaled[i]:>12.1%}{bare_scaled[i]:>12.1%}"
          f"{tmpl_scaled[i]:>12.1%}{ens_scaled[i]:>12.1%}")

## Image-to-image similarity

If we discard the text entirely, we can also use our similarity method to assess visual similarity between images. This is like finding nearest-neighbors.  We embed a query image and rank the other images by cosine similarity. This is the basis of reverse image search and deduplication.

In [ ]:
query_name = "astronaut"
query_vec = img_vecs[names.index(query_name)].unsqueeze(0)   # (1, dim)

scores = (img_vecs @ query_vec.T).squeeze(1).cpu().numpy()
print(f"Most similar to '{query_name}':")
for i in scores.argsort()[::-1]:
    tag = "  (self)" if names[i] == query_name else ""
    print(f"    {scores[i]:.3f}  {names[i]}{tag}")

## 10. Alternative: open_clip

If you want LAION-trained models or many architectures, `open_clip` exposes the same image/text-embedding interface. The API differs slightly but the mental model is identical.

In [ ]:
# !pip install -q open_clip_torch

In [ ]:
import open_clip, torch
from PIL import Image
from skimage import data

In [ ]:
model, _, preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32", pretrained="laion2b_s34b_b79k"
)
tokenizer = open_clip.get_tokenizer("ViT-B-32")
model.eval()

In [ ]:
image = preprocess(Image.fromarray(data.chelsea())).unsqueeze(0)
labels = ["a cat", "a dog"]
text = tokenizer(labels)

In [ ]:
with torch.no_grad():
    img_f = torch.nn.functional.normalize(model.encode_image(image), dim=-1)
    txt_f = torch.nn.functional.normalize(model.encode_text(text), dim=-1)
    probs = (100.0 * img_f @ txt_f.T).softmax(dim=-1).squeeze()

for label, p in sorted(zip(labels, probs), key=lambda x: -x[1]):
    print(f"{p:6.3%}  {label}")

## Final note

CLIP puts images and text in a shared, L2-normalized space where cosine similarity equals semantic match. Everything above is a rearrangement of that — classification compares one image to many texts, search compares one text to many images, dedup compares image to image.

**Limitations:**

- It's an embedding model, not generative — it scores and ranks, it can't describe or answer questions. (That's what the vision-chat API in the other notebook is for.)
- Accuracy is sensitive to prompt wording; use templates/ensembling.
- It struggles with counting, fine spatial relationships, text-in-images, and rare/specialized domains.
- It inherits biases from web-scraped training data — be cautious deploying it for classification of people.

**Performance tips:**

- Embed your image corpus once and cache the vectors; only the query needs encoding at search time. Vectors are tiny (512-1024 floats each).
- For large corpora, put the cached vectors in a vector index (FAISS, hnswlib) instead of a brute-force matrix multiply.
- Batch your `processor(...)` calls and run on GPU if available.